In [ ]:
import os, glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor


# =========================
# 1) Config (match your notebook)
# =========================
class CFG:
    data_dir     = "/content"     # <-- change to your folder
    csv_pattern  = "*.csv"             # your notebook used glob pattern
    in_dim       = 100                 # geometry bits
    out_dim      = 61                  # S11 points
    test_size    = 0.2
    seed         = 42

    # LightGBM hyperparams (good starting point)
    lgb_params = dict(
        n_estimators=5000,          # large + early stopping
        learning_rate=0.03,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.0,
        reg_lambda=1.0,
        random_state=seed,
        n_jobs=-1
    )

    early_stopping_rounds = 200
    eval_metric = "rmse"

    save_dir = "/content/lgbm_models"
    os.makedirs(save_dir, exist_ok=True)

# Frequency axis (61 points from 1 to 6 GHz)
freq_ghz = np.linspace(1.0, 6.0, CFG.out_dim)




In [ ]:
# =========================
# 2) Load data (same style as your notebook: folder + glob + concat)
# =========================
csv_files = sorted(glob.glob(os.path.join(CFG.data_dir, CFG.csv_pattern)))
print("Found CSV files:", csv_files)

if len(csv_files) == 0:
    raise FileNotFoundError(
        f"No CSV files found in {CFG.data_dir}. "
        "Upload your data and update CFG.data_dir."
    )

df_list = []
for f in csv_files:
    df_part = pd.read_csv(f, header=None)  # no header assumed
    df_list.append(df_part)

df = pd.concat(df_list, axis=0, ignore_index=True)
print("Combined data shape:", df.shape)

expected_cols = CFG.in_dim + CFG.out_dim
if df.shape[1] != expected_cols:
    raise ValueError(f"Expected {expected_cols} columns (100 geom + 61 S11), got {df.shape[1]}")

# Split
X = df.iloc[:, :CFG.in_dim].to_numpy()
Y = df.iloc[:, CFG.in_dim:CFG.in_dim + CFG.out_dim].to_numpy()

# Enforce binary on X (safe guard)
X = (X > 0.5).astype(np.int8)

print("X shape:", X.shape, "Y shape:", Y.shape)
print("X sample (first 20 bits):", X[0, :20])
print("Y sample (first 5 S11 pts):", Y[0, :5])




In [ ]:
# =========================
# 3) Train/Val split
# =========================
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=CFG.test_size, random_state=CFG.seed, shuffle=True
)

print("Train:", X_train.shape, Y_train.shape)
print("Val:  ", X_val.shape, Y_val.shape)




In [ ]:
# =========================
# 4) Train 61 separate LightGBM regressors
# =========================
models = []
train_rmse = np.zeros(CFG.out_dim, dtype=float)
val_rmse   = np.zeros(CFG.out_dim, dtype=float)
val_mae    = np.zeros(CFG.out_dim, dtype=float)

for k in range(CFG.out_dim):
    ytr = Y_train[:, k]
    yva = Y_val[:, k]

    model = LGBMRegressor(**CFG.lgb_params)

    model.fit(
        X_train, ytr,
        eval_set=[(X_val, yva)],
        eval_metric=CFG.eval_metric,
        callbacks=[],
    )

    # If your LightGBM version supports early stopping via callbacks, use this instead:
    # from lightgbm import early_stopping, log_evaluation
    # model.fit(
    #     X_train, ytr,
    #     eval_set=[(X_val, yva)],
    #     eval_metric=CFG.eval_metric,
    #     callbacks=[early_stopping(CFG.early_stopping_rounds), log_evaluation(0)]
    # )

    # Predict + metrics
    ytr_pred = model.predict(X_train)
    yva_pred = model.predict(X_val)

    # Fix: Removed 'squared=False' and applied np.sqrt() for RMSE calculation
    train_rmse[k] = np.sqrt(mean_squared_error(ytr, ytr_pred))
    val_rmse[k]   = np.sqrt(mean_squared_error(yva, yva_pred))
    val_mae[k]    = mean_absolute_error(yva, yva_pred)

    models.append(model)

    if (k % 10 == 0) or (k == CFG.out_dim - 1):
        print(f"[{k:02d}/{CFG.out_dim-1:02d}] Train RMSE={train_rmse[k]:.4f} | Val RMSE={val_rmse[k]:.4f}")

# Save all models
model_path = os.path.join(CFG.save_dir, "lgbm_models_61.joblib")
joblib.dump(models, model_path)
print("Saved models to:", model_path)




In [ ]:

# =========================
# 5) Evaluate: plot error vs frequency
# =========================
plt.figure()
plt.plot(freq_ghz, train_rmse, label="Train RMSE")
plt.plot(freq_ghz, val_rmse, label="Val RMSE")
plt.xlabel("Frequency (GHz)")
plt.ylabel("RMSE (dB)")
plt.title("LightGBM RMSE per S11 Frequency Point")
plt.grid(True)
plt.legend()
plt.show()

plt.figure()
plt.plot(freq_ghz, val_mae, label="Val MAE")
plt.xlabel("Frequency (GHz)")
plt.ylabel("MAE (dB)")
plt.title("LightGBM MAE per S11 Frequency Point")
plt.grid(True)
plt.legend()
plt.show()




In [ ]:

# =========================
# 6) Predict a full S11 curve (61 points) for any geometry
# =========================
def predict_s11(geometry_bits_100, models_61=models):
    """
    geometry_bits_100: list/np.array length 100 with 0/1
    returns: np.array length 61 (predicted S11 in dB)
    """
    x = np.asarray(geometry_bits_100).reshape(1, -1)
    if x.shape[1] != CFG.in_dim:
        raise ValueError(f"Expected geometry length {CFG.in_dim}, got {x.shape[1]}")
    x = (x > 0.5).astype(np.int8)

    yhat = np.array([m.predict(x)[0] for m in models_61], dtype=float)
    return yhat




In [ ]:
# =========================
# 7) Show one example curve (random val sample)
# =========================
idx = np.random.randint(0, X_val.shape[0])
y_true = Y_val[idx]
y_pred = predict_s11(X_val[idx])

plt.figure()
plt.plot(freq_ghz, y_true, label="True")
plt.plot(freq_ghz, y_pred, label="Predicted")
plt.xlabel("Frequency (GHz)")
plt.ylabel("S11 (dB)")
plt.title("Example: True vs Predicted S11 Curve (LightGBM)")
plt.grid(True)
plt.legend()
plt.show()

print("Example curve metrics:")
print("Curve RMSE (dB):", mean_squared_error(y_true, y_pred, squared=False))
print("Curve MAE (dB): ", mean_absolute_error(y_true, y_pred))